## script to wrangle Xenium data: parse cells.zarr.zip file to generate
1. cell segmentation
2. nuclei segmentation
3. cell x, y, z coordinate, we only takes x and y since the z levels are very very close

An optional retangle qupath annotation file can be provided to focus on cells in the rectangle, needs
- geojson file

In [51]:
import os
import zarr
import numpy as np
import cv2
import pandas as pd
import json

## xenium cells.zarr information
https://www.10xgenomics.com/support/software/xenium-onboard-analysis/latest/analysis/xoa-output-zarr#cells

In [52]:
xenium_bundle_dir = "/mnt/windows/16-080L/Xenium/output-XETG00049__0015980__XE009-80-C__20231220__211433/"
cell_zarr_file = "cells.zarr.zip"
cell_zarr_file = os.path.join(xenium_bundle_dir, cell_zarr_file)
edge_color = 200
output_path_cell_edge = "cell_edge.tif"
output_path_nuc_edge = "nuc_edge.tif"
output_path_cells = "cells.tsv"
geojsonfile = None

## Read .zarr files

In [53]:
root = zarr.open(cell_zarr_file, mode='r')

In [54]:
# Look at group array info and structure
root.info, root.tree(), sorted(root.attrs), root.attrs["polygon_set_descriptions"],root.attrs["spatial_units"] # shows structure, array dimensions, data types

(Name        : /
 Type        : zarr.hierarchy.Group
 Read-only   : True
 Store type  : zarr.storage.ZipStore
 No. members : 6
 No. arrays  : 5
 No. groups  : 1
 Arrays      : cell_id, cell_summary, polygon_num_vertices, polygon_vertices,
             : seg_mask_value
 Groups      : masks,
 /
  ├── cell_id (334055, 2) uint32
  ├── cell_summary (334055, 7) float64
  ├── masks
  │   ├── 0 (71432, 39859) uint32
  │   ├── 1 (71432, 39859) uint32
  │   └── homogeneous_transform (4, 4) float32
  ├── polygon_num_vertices (2, 334055) int32
  ├── polygon_vertices (2, 334055, 26) float32
  └── seg_mask_value (334055,) uint32,
 ['major_version',
  'minor_version',
  'name',
  'number_cells',
  'polygon_set_descriptions',
  'polygon_set_display_names',
  'polygon_set_names',
  'spatial_units'],
 ['DAPI-based nuclei segmentation', 'Expansion of nuclei boundaries by 15 μm'],
 'microns')

In [55]:
# Create cell and nucleus segmentation mask np array objects to read or modify
cellseg_mask = np.array(root["masks"][1])
nucseg_mask = np.array(root["masks"][0])

In [56]:
cellseg_mask.shape, nucseg_mask.shape

((71432, 39859), (71432, 39859))

In [57]:
cellseg_mask.max(), nucseg_mask.max()

(334055, 334055)

## (optional) Read  geojsonfile (from Qupath single rectangle annotation)

In [58]:
geojsonfile = "morphology_16-080L.geojson"
px_to_um = 0.2125
J = json.load(open(geojsonfile, 'r'))
rectangle = pd.DataFrame(J["features"][0]["geometry"]["coordinates"][0])
# x
x_min = min(rectangle[0]) * px_to_um
x_max = max(rectangle[0]) * px_to_um
# y
y_min = min(rectangle[1]) * px_to_um
y_max = max(rectangle[1]) * px_to_um

## Convert mask to edge

In [87]:
def edgePixel (image, row, column, rows, columns):
    if row % 1000 ==0 and column == columns -1:
        print (row, column)
    if image[row, column] == 0:
        return 0
    if row > 0 and image[row, column]!= image[row -1, column]:
        return 1
    if row < rows - 1 and image[row, column]!= image[row +1, column]:
        return 1
    if column > 0 and image[row, column]!= image[row, column -1]:
        return 1
    if column < columns - 1 and image[row, column]!= image[row, column +1]:
        return 1
    return 0

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from functools import partial
import multiprocessing

# Specify the desired number of CPUs
num_cpus =  multiprocessing.cpu_count()

rows, columns = cellseg_mask.shape

# Use ThreadPoolExecutor for parallel processing with a specified number of CPUs
with ProcessPoolExecutor(max_workers=num_cpus) as executor:
    # Define a function to process a single element
    def process_row_cellseg_mask(args):
        i, row = args
        result = []
        for j in range (0, columns):
            result.append(edgePixel (cellseg_mask, i, j, rows, columns))
        return result
        
    def process_row_nucseg_mask(args):
        i, row = args
        result = []
        for j in range (0, columns):
            result.append(edgePixel (nucseg_mask, i, j, rows, columns))
        return result
        
    # Use executor.map to apply the function to each element in parallel
    cell_edge_array = np.array(list(executor.map(process_row_cellseg_mask, list(enumerate(cellseg_mask)))))
    nuc_edge_array = np.array(list(executor.map(process_row_nucseg_mask, list(enumerate(nucseg_mask)))))

In [ ]:
cell_edge_array, cell_edge_array.shape, nuc_edge_array, nuc_edge_array.shape

In [90]:
cell_edge_arrayCopy = cell_edge_array.astype(np.uint8)
nuc_edge_arrayCopy = nuc_edge_array.astype(np.uint8)

In [91]:
cell_edge_arrayCopy, cell_edge_arrayCopy.shape, nuc_edge_arrayCopy, nuc_edge_arrayCopy.shape

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=uint8),
 (71432, 39859),
 array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=uint8),
 (71432, 39859))

In [92]:
cell_edge_arrayCopy[cell_edge_arrayCopy != 0] = edge_color
nuc_edge_arrayCopy[nuc_edge_arrayCopy != 0] = edge_color

In [204]:
cv2.imwrite(output_path_cell_edge, cell_edge_arrayCopy)
cv2.imwrite(output_path_nuc_edge, nuc_edge_arrayCopy)

True

### (optinal) cut image to match rectangle

In [94]:
geojsonfile = "morphology_16-080L.geojson"
J = json.load(open(geojsonfile, 'r'))
rectangle = pd.DataFrame(J["features"][0]["geometry"]["coordinates"][0])
# x
x_min = min(rectangle[0]) 
x_max = max(rectangle[0])
# y
y_min = min(rectangle[1]) 
y_max = max(rectangle[1])

In [97]:
cv2.imwrite(output_path_cell_edge, cell_edge_arrayCopy[y_min:y_max, x_min:x_max])
cv2.imwrite(output_path_nuc_edge, nuc_edge_arrayCopy[y_min:y_max, x_min:x_max])

True

## Build cells.tsv including x,y,z coorodinates

In [74]:
root["cell_summary"], root["cell_id"]

(<zarr.core.Array '/cell_summary' (334055, 7) float64 read-only>,
 <zarr.core.Array '/cell_id' (334055, 2) uint32 read-only>)

In [75]:
def convert_to_xenium_cell_id (args):
    cell_id_prefix, dataset_suffix = args
    # print (cell_id_prefix, hex(cell_id_prefix)[2:])
    #  [0 - 9, a - f] to the range a - p
    hex_to_xenium = {
        "0":"a",
        "1":"b",
        "2":"c",
        "3":"d",
        "4":"e",
        "5":"f",
        "6":"g",
        "7":"h",
        "8":"i",
        "9":"j",
        "a":"k",
        "b":"l",
        "c":"m",
        "d":"n",
        "e":"o",
        "f":"p"
    }
    def convert_hex_to_xenium(hex):
        return hex_to_xenium[hex]
        
    cell_id_prefix_xenium = "".join(list(map(convert_hex_to_xenium, hex(cell_id_prefix)[2:])))
    cell_id_prefix_xenium = "a" * (8 - len(cell_id_prefix_xenium)) + cell_id_prefix_xenium
    cell_id =  cell_id_prefix_xenium + "-" + str(dataset_suffix)
    return cell_id

In [76]:
xenium_cell_id = list(map(convert_to_xenium_cell_id, list(zip(root["cell_id"][:,0], root["cell_id"][:,1]))))

In [77]:
df = pd.DataFrame(root["cell_summary"], 
                  columns = ["cell_centroid_x", "cell_centroid_y", "cell_area", "nucleus_centroid_x","nucleus_centroid_y", "nucleus_area", "z_level"], 
                  index = xenium_cell_id)

In [78]:
df

,cell_centroid_x,cell_centroid_y,cell_area,nucleus_centroid_x,nucleus_centroid_y,nucleus_area,z_level
aaaaadfk-1,822.971619,7299.142578,353.708919,826.048523,7305.054688,28.854845,21.0
aaaabcck-1,828.437805,7333.003418,217.924070,827.635437,7335.468750,4.605938,24.0
aaaaidhj-1,840.336853,7343.100586,100.879066,841.786987,7344.538086,16.888438,21.0
aaaaiheo-1,809.179138,7597.605469,158.137193,809.072632,7594.875488,14.540313,21.0
aaaaokan-1,832.967529,7317.130371,55.948596,833.934509,7313.253906,5.734844,18.0
...,...,...,...,...,...,...,...
oijcibae-1,7508.504395,886.216248,143.867818,7509.102539,886.409424,38.789220,24.0
oijcjhcf-1,7516.332031,883.460083,142.784068,7513.018066,882.607910,34.363907,21.0
oijddlfb-1,7511.994141,875.118286,68.682659,7513.358398,874.333252,14.901563,18.0
oijdemdk-1,7770.033691,10998.489258,1001.023786,7770.152344,10998.171875,28.538751,30.0


In [79]:
max(df.cell_centroid_x), max(df.cell_centroid_y)

(8348.8681640625, 15170.015625)

In [80]:
if geojsonfile:
    print(x_min, x_max, y_min, y_max)
    selected_df = df[(df.cell_centroid_x> x_min) 
                    & (df.cell_centroid_x < x_max) 
                    & (df.cell_centroid_y > y_min)
                    & (df.cell_centroid_y < y_max)
    ]
else:
    selected_df = df
    print("no rectangle annotation")


1322.3875 2862.5875 9901.4375 14595.9875


In [81]:
selected_df

,cell_centroid_x,cell_centroid_y,cell_area,nucleus_centroid_x,nucleus_centroid_y,nucleus_area,z_level
aadomelj-1,1648.644409,12063.889648,280.736416,1644.863281,12067.308594,92.525160,27.0
aadooijh-1,1642.802979,12074.333984,132.443286,1642.784424,12073.924805,103.046566,27.0
aadpffob-1,1656.759155,12070.237305,80.829690,1652.790405,12072.628906,13.005000,24.0
aadpkdac-1,1671.749756,12067.138672,315.551886,1672.067871,12074.653320,45.923908,27.0
aadplnnc-1,1667.880981,12078.832031,64.257346,1668.926270,12079.154297,40.234220,27.0
...,...,...,...,...,...,...,...
nhjlofbn-1,2520.729736,11047.266602,41.227658,2520.143311,11047.365234,32.286720,27.0
nhjmapda-1,2503.906006,11052.137695,95.686097,2503.577881,11052.268555,64.844377,27.0
nhjmcomn-1,2521.197510,11053.026367,52.561877,2520.817871,11052.836914,39.511720,27.0
nhjmiljo-1,2513.402588,11052.323242,43.124220,2513.627686,11052.429688,33.235001,27.0


In [174]:
selected_df.to_csv(output_path_cells, sep="\t")

## sepcific for tissue microarray

In [82]:
ylim1 = 11400
ylim2 = 13000
selected_df.loc[:,"core"]= ""
selected_df.loc[selected_df.cell_centroid_y < ylim1, "core"] = "16-080L1 C1"
selected_df.loc[(selected_df.cell_centroid_y > ylim1) & (selected_df.cell_centroid_y < ylim2), "core"] = "16-080L1 C2"
selected_df.loc[selected_df.cell_centroid_y > ylim2, "core"] = "16-080L1 C3"

/tmp/ipykernel_4055545/3932278376.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df.loc[:,"core"]= ""


In [83]:
selected_df

,cell_centroid_x,cell_centroid_y,cell_area,nucleus_centroid_x,nucleus_centroid_y,nucleus_area,z_level,core
aadomelj-1,1648.644409,12063.889648,280.736416,1644.863281,12067.308594,92.525160,27.0,16-080L1 C2
aadooijh-1,1642.802979,12074.333984,132.443286,1642.784424,12073.924805,103.046566,27.0,16-080L1 C2
aadpffob-1,1656.759155,12070.237305,80.829690,1652.790405,12072.628906,13.005000,24.0,16-080L1 C2
aadpkdac-1,1671.749756,12067.138672,315.551886,1672.067871,12074.653320,45.923908,27.0,16-080L1 C2
aadplnnc-1,1667.880981,12078.832031,64.257346,1668.926270,12079.154297,40.234220,27.0,16-080L1 C2
...,...,...,...,...,...,...,...,...
nhjlofbn-1,2520.729736,11047.266602,41.227658,2520.143311,11047.365234,32.286720,27.0,16-080L1 C1
nhjmapda-1,2503.906006,11052.137695,95.686097,2503.577881,11052.268555,64.844377,27.0,16-080L1 C1
nhjmcomn-1,2521.197510,11053.026367,52.561877,2520.817871,11052.836914,39.511720,27.0,16-080L1 C1
nhjmiljo-1,2513.402588,11052.323242,43.124220,2513.627686,11052.429688,33.235001,27.0,16-080L1 C1


In [50]:
selected_df.to_csv(output_path_cells, sep="\t")

## sepcific for shifting cell cooridnates to match image cut to rectangle

In [84]:
selected_df.loc[:,"cell_centroid_x"]= selected_df["cell_centroid_x"] - x_min
selected_df.loc[:,"cell_centroid_y"]= selected_df["cell_centroid_y"] - y_min
selected_df.loc[:,"nucleus_centroid_x"]= selected_df["nucleus_centroid_x"] - x_min
selected_df.loc[:,"nucleus_centroid_y"]= selected_df["nucleus_centroid_y"] - y_min

In [85]:
selected_df

,cell_centroid_x,cell_centroid_y,cell_area,nucleus_centroid_x,nucleus_centroid_y,nucleus_area,z_level,core
aadomelj-1,326.256909,2162.452148,280.736416,322.475781,2165.871094,92.525160,27.0,16-080L1 C2
aadooijh-1,320.415479,2172.896484,132.443286,320.396924,2172.487305,103.046566,27.0,16-080L1 C2
aadpffob-1,334.371655,2168.799805,80.829690,330.402905,2171.191406,13.005000,24.0,16-080L1 C2
aadpkdac-1,349.362256,2165.701172,315.551886,349.680371,2173.215820,45.923908,27.0,16-080L1 C2
aadplnnc-1,345.493481,2177.394531,64.257346,346.538770,2177.716797,40.234220,27.0,16-080L1 C2
...,...,...,...,...,...,...,...,...
nhjlofbn-1,1198.342236,1145.829102,41.227658,1197.755811,1145.927734,32.286720,27.0,16-080L1 C1
nhjmapda-1,1181.518506,1150.700195,95.686097,1181.190381,1150.831055,64.844377,27.0,16-080L1 C1
nhjmcomn-1,1198.810010,1151.588867,52.561877,1198.430371,1151.399414,39.511720,27.0,16-080L1 C1
nhjmiljo-1,1191.015088,1150.885742,43.124220,1191.240186,1150.992188,33.235001,27.0,16-080L1 C1


In [86]:
selected_df.to_csv(output_path_cells, sep="\t")